In [1]:
from zhinst.toolkit import Session, Sequence, Waveforms
from zhinst.toolkit.waveform import Wave, OutputType
import numpy as np
import time

ModuleNotFoundError: No module named 'zhinst'

In [2]:
def comp_upload_seq(awgnode,code,device,session):

    awg_node = device.awgs[awgnode]
    awg = session.modules.awg
    awg.device(device.serial)
    awg.index(awgnode)
    awg.sequencertype('sg' if device.device_type.startswith('SHFSG') or device.device_type.startswith('SHFQC') else 'auto-detect')
    awg.execute()
    awg.compiler.sourcestring(code)

    # The following lines are not mandatory but only to ensure that everything was compiled and uploaded correctly. 
    timeout = 100.0  # seconds
    compiler_status = awg.compiler.status()
    start = time.time()
    while compiler_status == -1:
        if time.time() - start >= timeout:
            raise TimeoutError("Program compilation timed out")
        time.sleep(0.1)
        compiler_status = awg.compiler.status()
        if compiler_status == 1:
            print(awg.compiler.statusstring())
            raise RuntimeError(
                "Error during sequencer compilation. Check the log for detailed information"
            )
        if compiler_status == 2:
            print(f"Warning during sequencer compilation {awg.compiler.statusstring()}")
    # Check and wait until the elf upload to the device was successful
    print("Sequence loaded.")
    progress = awg.progress()
    while progress < 1.0 or awg.elf.status() == 2 or awg_node.ready() == 0:
        if time.time() - start >= timeout:
            raise TimeoutError(f"Program upload timed out")
        time.sleep(0.1)
        progress = awg.progress()
    if awg.elf.status() or not awg_node.ready():
        raise RuntimeError(
            "Error during upload of ELF file. Check the log for detailed information"
        )
    print("AWG ready for data upload.")

    return

def add_truncgauss_risefall(wave,risefall_t,sclk):
    '''
    Given a waveform with unit amplitude and a rise-fall time, this returns the same wave with 
    a truncated gaussian envelope with a flat top.
    '''
    def trunc_gauss(x,tau,sigma):
        return ((np.exp(-0.5*((x-tau/2)/sigma)**2) - np.exp(-0.5*((tau/2)/sigma)**2))/(1-np.exp(-0.5*((tau/2)/sigma)**2)))

    tau = int(risefall_t*sclk)*2 #rise + fall time in points
    if (tau) > wave.size : raise Exception("Wave too short for complete rise and fall.")
    sigma = tau/4 # of the gaussian
    #calculate envelope
    truncgaussarr = np.array([trunc_gauss(i,tau,sigma) for i in range(tau)])
    #apply envelope
    wavef = np.copy(wave)
    wavef[:int(tau/2)] = np.multiply(wave[:int(tau/2)],truncgaussarr[:int(tau/2)])
    wavef[-int(tau/2):] = np.multiply(wave[-int(tau/2):],truncgaussarr[-int(tau/2):])
    return wavef

def gen_sin(freq,phase,amp,len,sclk):
    '''
    Generate a sin wave with frequency in Hz, phase in radians, length in seconds, sclk in samples/sec
    '''
    no_pts = int(len*sclk)
    omega_pt = freq/sclk
    wave = amp*np.sin([(omega_pt*i+phase) for i in range(no_pts)])
    return wave

def round_off_wvf_to_16_wvfpoints(wavf,front=False,back=False):
    if wavf.size <= 32 : return np.hstack((np.zeros(32-wavf.size),wavf))
    if back == True:
        return np.hstack((wavf,np.zeros(16-int(wavf.size%16))))
    if front == True:
        return np.hstack((np.zeros(16-int(wavf.size%16)),wavf))



In [3]:
session = Session("localhost")
session.devices.visible()

NameError: name 'Session' is not defined

In [4]:
device = session.connect_device("dev8772")

NameError: name 'session' is not defined